In [ ]:
# import libraries
try:
  # %tensorflow_version only exists in Colab.
  !pip install tf-nightly
except Exception:
  pass
import tensorflow as tf
import pandas as pd
from tensorflow import keras
!pip install tensorflow-datasets
import tensorflow_datasets as tfds
import numpy as np
import matplotlib.pyplot as plt

print(tf.__version__)

In [ ]:
# get data files
!wget https://cdn.freecodecamp.org/project-data/sms/train-data.tsv
!wget https://cdn.freecodecamp.org/project-data/sms/valid-data.tsv

train_file_path = "train-data.tsv"
test_file_path = "valid-data.tsv"

In [ ]:
from tensorflow.keras import layers
# Load and prepare data
train_df = pd.read_csv(train_file_path, sep='\t', header=None, names=['label','text'])
valid_df = pd.read_csv(test_file_path, sep='\t', header=None, names=['label','text'])

train_df['label_num'] = train_df['label'].map({'ham':0, 'spam':1})
valid_df['label_num'] = valid_df['label'].map({'ham':0, 'spam':1})

train_texts = train_df['text'].astype(str).values
train_labels = train_df['label_num'].values
valid_texts = valid_df['text'].astype(str).values
valid_labels = valid_df['label_num'].values

# Text vectorization - converts strings to integer sequences
VOCAB_SIZE = 10000
SEQUENCE_LENGTH = 100
vectorize_layer = layers.TextVectorization(
    max_tokens=VOCAB_SIZE,
    output_mode='int',
    output_sequence_length=SEQUENCE_LENGTH
)
vectorize_layer.adapt(train_texts)

# Model: Embedding + Bidirectional LSTM + Dense
model = keras.Sequential([
    vectorize_layer,
    layers.Embedding(VOCAB_SIZE, 16, mask_zero=True),
    layers.Bidirectional(layers.LSTM(32)),
    layers.Dense(32, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(1, activation='sigmoid')
])

model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])

# Train - account for imbalance: 3619 ham vs 560 spam (~1:6.46)
history = model.fit(
    train_texts, train_labels,
    epochs=12,
    batch_size=32,
    validation_data=(valid_texts, valid_labels),
    class_weight={0: 1, 1: 6.46},
    verbose=1
)

In [ ]:
# function to predict messages based on model
# (should return list containing prediction and label, ex. [0.008318834938108921, 'ham'])
def predict_message(pred_text):
    prediction = model.predict(tf.constant([pred_text]), verbose=0)[0][0]
    label = "spam" if prediction > 0.5 else "ham"
    return [float(prediction), label]

pred_text = "how are you doing today?"
prediction = predict_message(pred_text)
print(prediction)

In [ ]:
# Run this cell to test your function and model. Do not modify contents.
def test_predictions():
  test_messages = ["how are you doing today",
                   "sale today! to stop texts call 98912460324",
                   "i dont want to go. can we try it a different day? available sat",
                   "our new mobile video service is live. just install on your phone to start watching.",
                   "you have won £1000 cash! call to claim your prize.",
                   "i'll bring it tomorrow. don't forget the milk.",
                   "wow, is your arm alright. that happened to me one time too"
                  ]

  test_answers = ["ham", "spam", "ham", "spam", "spam", "ham", "ham"]
  passed = True

  for msg, ans in zip(test_messages, test_answers):
    prediction = predict_message(msg)
    if prediction[1] != ans:
      passed = False

  if passed:
    print("You passed the challenge. Great job!")
  else:
    print("You haven't passed yet. Keep trying.")

test_predictions()
